# Mastering Modern Java: From Functional to Advanced (Java 8 to 25)

**Interactive Notebook - Enhanced Edition**

This notebook lets you run all training examples directly in your browser via Google Colab.

| Module | Topics | Java Version |
|--------|--------|--------------|
| 1 | Functional Interfaces & Lambdas | 8+ |
| 2 | Stream API | 8-16+ |
| 3 | Records, Sealed Classes, Pattern Matching | 14-25 |
| 4 | Virtual Threads & Concurrency | 21-24 |
| 5 | Unnamed Variables & Markdown JavaDoc | 22-23 |
| 6 | Stream Gatherers & Scoped Values | 24-25 |
| 7 | Java 25 Language Features | 25 LTS |

> **How it works**: We install JDK 25, then each code cell writes a `.java` file, compiles it, and runs it.

---
## Setup: Install JDK 25

Run this cell **once** at the start of your session. It installs OpenJDK 25 via SDKMAN so all modules (1-7) work.

In [ ]:
%%bash
# Install SDKMAN and JDK 25
export SDKMAN_DIR="$HOME/.sdkman"
if [ ! -d "$SDKMAN_DIR" ]; then
  curl -s "https://get.sdkman.io?rcupdate=false" | bash > /dev/null 2>&1
fi
source "$SDKMAN_DIR/bin/sdkman-init.sh"
sdk install java 25-tem > /dev/null 2>&1

# Create a wrapper so !java and !javac use JDK 25
JAVA_HOME=$(sdk home java 25-tem)
ln -sf "$JAVA_HOME/bin/java" /usr/local/bin/java
ln -sf "$JAVA_HOME/bin/javac" /usr/local/bin/javac

echo "Setup complete!"
java --version

---
# Module 1: Functional Interfaces & Lambdas (Java 8+)

**Goal:** Transition from verbose anonymous inner classes to concise, readable, and functional code.

## 1.1 What is a Functional Interface?

A functional interface has **exactly one abstract method**. This allows it to be the target for a lambda or method reference.

The `@FunctionalInterface` annotation is optional but recommended.

## 1.2 The "Core Four" Functional Interfaces

| Interface | Signature | Purpose |
|---|---|---|
| `Predicate<T>` | `boolean test(T t)` | Test a condition (filter/match) |
| `Function<T,R>` | `R apply(T t)` | Transform data |
| `Consumer<T>` | `void accept(T t)` | Perform side effect |
| `Supplier<T>` | `T get()` | Produce/provide a value |

In [ ]:
%%writefile CoreFourLab.java
import java.util.function.*;

public class CoreFourLab {
    public static void main(String[] args) {
        // === PREDICATE: test a condition ===
        // Before: Anonymous Class
        Predicate<String> isLongOld = new Predicate<String>() {
            @Override
            public boolean test(String s) { return s.length() > 10; }
        };
        // After: Lambda
        Predicate<String> isLong = s -> s.length() > 10;
        System.out.println("'hello' is long? " + isLong.test("hello"));
        System.out.println("'introduction' is long? " + isLong.test("introduction"));

        // === FUNCTION: transform data ===
        Function<String, Integer> getLength = s -> s.length();
        System.out.println("\nLength of 'world': " + getLength.apply("world"));

        // === CONSUMER: side effect ===
        Consumer<String> printUpper = s -> System.out.println(s.toUpperCase());
        System.out.print("\nConsumer output: ");
        printUpper.accept("hello world");

        // === SUPPLIER: produce value ===
        Supplier<String> greeting = () -> "Hello from Supplier!";
        System.out.println("\n" + greeting.get());
    }
}

In [ ]:
!javac CoreFourLab.java && java CoreFourLab

## 1.3 Method References (`::`)

Method references are shorthand for lambdas that call an existing method:

| Type | Syntax | Lambda equivalent |
|---|---|---|
| Static | `ClassName::staticMethod` | `x -> ClassName.staticMethod(x)` |
| Instance (bound) | `instance::method` | `x -> instance.method(x)` |
| Instance (unbound) | `ClassName::method` | `(obj, x) -> obj.method(x)` |
| Constructor | `ClassName::new` | `x -> new ClassName(x)` |

In [ ]:
%%writefile MethodRefLab.java
import java.util.List;
import java.util.stream.Collectors;

public class MethodRefLab {
    public static void main(String[] args) {
        List<String> names = List.of("anna", "bob", "charlie");

        System.out.println("=== forEach with method reference ===");
        names.forEach(System.out::println);

        List<String> upper = names.stream()
            .map(String::toUpperCase)
            .collect(Collectors.toList());
        System.out.println("\nUppercase: " + upper);
    }
}

In [ ]:
!javac MethodRefLab.java && java MethodRefLab

---
# Module 2: The Stream API - Mastering Data Pipelines (Java 8+)

**Goal**: Shift from imperative loops ("how to do it") to declarative pipelines ("what to do").

## 2.1 Lazy vs. Eager: The Core of Stream Performance

Intermediate operations are **lazy** - they do nothing until a terminal operation triggers the pipeline.

In [ ]:
%%writefile LazyStreamLab.java
import java.util.List;
import java.util.stream.Stream;

public class LazyStreamLab {
    public static void main(String[] args) {
        List<String> names = List.of("Alice", "Bob", "Charlie", "David");

        System.out.println("Defining the stream...");
        Stream<String> stream = names.stream()
            .peek(name -> System.out.println("  Peeking at: " + name))
            .filter(name -> name.length() > 4);

        System.out.println("Stream defined, but NO output yet (lazy!).");
        System.out.println("\nCalling terminal operation findFirst()...");

        stream.findFirst();

        System.out.println("\nNotice: 'Bob' and 'David' were never peeked at!");
    }
}

In [ ]:
!javac LazyStreamLab.java && java LazyStreamLab

## 2.2 Key Stream Operations

### Intermediate (lazy):
`filter`, `map`, `flatMap`, `distinct`, `sorted`, `limit`, `skip`

### Terminal (trigger execution):
`forEach`, `collect`, `toList()` (Java 16+), `reduce`, `anyMatch`/`allMatch`/`noneMatch`, `findFirst`/`findAny`

> **`toList()` vs `Collectors.toList()`**: Since Java 16, `.toList()` is shorter but returns an **unmodifiable** list.

## 2.3 Deep Dive: `flatMap` - Flattening Nested Structures

In [ ]:
%%writefile FlatMapLab.java
import java.util.List;

public class FlatMapLab {

    record BlogPost(String title, List<String> tags) {}

    public static void main(String[] args) {
        List<BlogPost> posts = List.of(
            new BlogPost("Java Streams Guide", List.of("java", "streams", "functional")),
            new BlogPost("Spring Boot Tips", List.of("java", "spring", "backend")),
            new BlogPost("React with Java", List.of("react", "java", "fullstack"))
        );

        // map() gives Stream<List<String>> -- we want a flat Stream<String>
        List<String> allUniqueTags = posts.stream()
            .flatMap(post -> post.tags().stream())
            .distinct()
            .sorted()
            .toList();

        System.out.println("All unique tags: " + allUniqueTags);
    }
}

In [ ]:
!javac FlatMapLab.java && java FlatMapLab

## 2.4 Deep Dive: `reduce` - Combining Elements Into a Single Result

In [ ]:
%%writefile ReduceLab.java
import java.util.List;
import java.util.Optional;
import java.util.stream.Collectors;

public class ReduceLab {
    public static void main(String[] args) {
        List<Integer> numbers = List.of(1, 2, 3, 4, 5);

        int sum = numbers.stream().reduce(0, Integer::sum);
        System.out.println("Sum: " + sum);

        int product = numbers.stream().reduce(1, (a, b) -> a * b);
        System.out.println("Product: " + product);

        Optional<Integer> max = numbers.stream().reduce(Integer::max);
        max.ifPresent(m -> System.out.println("Max: " + m));

        // String joining with reduce vs Collectors.joining()
        List<String> words = List.of("Java", "is", "awesome");
        String sentence = words.stream().reduce((a, b) -> a + " " + b).orElse("");
        System.out.println("Sentence: " + sentence);

        String better = words.stream().collect(Collectors.joining(" "));
        System.out.println("Better (Collectors.joining): " + better);
    }
}

In [ ]:
!javac ReduceLab.java && java ReduceLab

## 2.5 Advanced Collectors: Grouping & Partitioning

In [ ]:
%%writefile CollectorLab.java
import java.util.List;
import java.util.Map;
import java.util.stream.Collectors;

public class CollectorLab {

    record Employee(String name, String department, int salary) {}

    public static void main(String[] args) {
        List<Employee> employees = List.of(
            new Employee("Alice", "Engineering", 120000),
            new Employee("Bob", "Engineering", 110000),
            new Employee("Charlie", "HR", 90000),
            new Employee("Diana", "HR", 95000),
            new Employee("Eve", "Sales", 80000)
        );

        Map<String, List<Employee>> byDept = employees.stream()
            .collect(Collectors.groupingBy(Employee::department));
        System.out.println("By department: " + byDept);

        Map<String, Double> avgSalary = employees.stream()
            .collect(Collectors.groupingBy(
                Employee::department,
                Collectors.averagingInt(Employee::salary)
            ));
        System.out.println("\nAvg salary by dept: " + avgSalary);

        Map<Boolean, List<Employee>> highEarners = employees.stream()
            .collect(Collectors.partitioningBy(e -> e.salary() > 100000));
        System.out.println("\nHigh earners (true) vs others: " + highEarners);
    }
}

In [ ]:
!javac CollectorLab.java && java CollectorLab

---
# Module 3: Modern Data Modeling (Java 14-25)

**Goal**: Eliminate boilerplate and create clear, robust, immutable data models.

## 3.1 Records (Java 16+)

A `record` is an immutable data carrier. Auto-generates `equals()`, `hashCode()`, `toString()`, and accessors.

In [ ]:
%%writefile RecordLab.java
public class RecordLab {

    record PositiveAmount(int value) {
        PositiveAmount {
            if (value < 0) throw new IllegalArgumentException("Amount cannot be negative");
        }
        PositiveAmount add(PositiveAmount other) {
            return new PositiveAmount(this.value + other.value);
        }
    }

    public static void main(String[] args) {
        var p1 = new PositiveAmount(100);
        var p2 = new PositiveAmount(50);
        System.out.println("Sum: " + p1.add(p2).value());
        System.out.println("toString: " + p1);
        System.out.println("equals: " + p1.equals(new PositiveAmount(100)));

        try { new PositiveAmount(-10); }
        catch (IllegalArgumentException e) { System.out.println("Caught: " + e.getMessage()); }
    }
}

In [ ]:
!javac RecordLab.java && java RecordLab

## 3.2 Sealed Classes (Java 17+) & Pattern Matching (Java 21)

**Sealed classes** define a *closed* hierarchy. **Pattern matching** in `switch` (Java 21) adds `when` guards.

In [ ]:
%%writefile PatternLab.java
public class PatternLab {

    sealed interface ApiResult permits Success, Failure {}
    record Success(String data) implements ApiResult {}
    record Failure(int errorCode, String message) implements ApiResult {}

    static String handleResult(ApiResult result) {
        return switch (result) {
            case Success s when s.data().contains("error") -> "Success contained error: " + s.data();
            case Success s -> "Data received: " + s.data();
            case Failure f when f.errorCode() == 404 -> "Resource not found.";
            case Failure f -> "Generic failure. Code: " + f.errorCode();
        };
    }

    public static void main(String[] args) {
        System.out.println(handleResult(new Success("{\"name\":\"John\"}")));
        System.out.println(handleResult(new Success("{\"status\":\"error\"}")));
        System.out.println(handleResult(new Failure(404, "Not Found")));
        System.out.println(handleResult(new Failure(500, "Server Error")));
    }
}

In [ ]:
!javac PatternLab.java && java PatternLab

## 3.3 Combining Streams with Pattern Matching

In [ ]:
%%writefile StreamPatternLab.java
import java.util.List;
import java.util.stream.Collectors;

public class StreamPatternLab {
    sealed interface ApiResult permits Success, Failure {}
    record Success(String data) implements ApiResult {}
    record Failure(int errorCode, String message) implements ApiResult {}

    public static void main(String[] args) {
        List<ApiResult> results = List.of(
            new Success("user-1"),
            new Failure(404, "user-2 not found"),
            new Success("user-3"),
            new Failure(500, "server error"),
            new Success("user-4")
        );

        List<String> successes = results.stream()
            .filter(r -> r instanceof Success)
            .map(r -> ((Success) r).data())
            .toList();
        System.out.println("Successes: " + successes);

        var errorSummary = results.stream()
            .filter(r -> r instanceof Failure)
            .map(r -> (Failure) r)
            .collect(Collectors.groupingBy(Failure::errorCode, Collectors.counting()));
        System.out.println("Error summary: " + errorSummary);

        System.out.println("\n--- Full Report (switch in .map) ---");
        results.stream()
            .map(r -> switch (r) {
                case Success s -> "[OK] " + s.data();
                case Failure f -> "[ERR " + f.errorCode() + "] " + f.message();
            })
            .forEach(System.out::println);
    }
}

In [ ]:
!javac StreamPatternLab.java && java StreamPatternLab

---
# Module 4: High-Performance Concurrency (Java 21-24)

**Goal**: Leverage Virtual Threads for simple, scalable concurrent code.

## 4.1 How Virtual Threads Work

1. A virtual thread runs on a "carrier" platform thread
2. On blocking I/O, it **unmounts** from the carrier
3. The carrier is free to run another virtual thread
4. When I/O completes, the virtual thread **mounts** back on any available carrier

> `Executors.newVirtualThreadPerTaskExecutor()` is **stable and final** since Java 21.

## 4.2 Virtual Threads in Practice

In [ ]:
%%writefile VirtualThreadLab.java
import java.time.Duration;
import java.time.Instant;
import java.util.concurrent.Executors;
import java.util.stream.IntStream;

public class VirtualThreadLab {
    public static void main(String[] args) {
        Instant start = Instant.now();

        try (var executor = Executors.newVirtualThreadPerTaskExecutor()) {
            IntStream.range(0, 10_000).forEach(i -> {
                executor.submit(() -> {
                    Thread.sleep(Duration.ofSeconds(1));
                    return i;
                });
            });
        }

        long elapsed = Duration.between(start, Instant.now()).toMillis();
        System.out.println("Completed 10,000 tasks in: " + elapsed + "ms");
        System.out.println("(Not 10,000 seconds! Virtual threads unmount during sleep.)");
    }
}

In [ ]:
!javac VirtualThreadLab.java && java VirtualThreadLab

## 4.3 Virtual Threads + `synchronized` Fix (Java 24)

Before Java 24, a virtual thread blocked inside `synchronized` would **pin** its carrier thread. Java 24 (JEP 491) fixes this transparently — no code change needed.

```java
// Before Java 24: this could cause thread starvation
// After Java 24: works correctly
synchronized (lock) {
    var result = httpClient.send(request, BodyHandlers.ofString());
}
// Tip: On Java 21-23, prefer ReentrantLock. On Java 24+, synchronized is fine.
```

## 4.4 Structured Concurrency (Preview)

> `StructuredTaskScope` remains a **preview feature** through Java 25 (JEP 505). The API has evolved across versions. This example uses JDK 21 preview syntax.

In [ ]:
%%writefile StructuredLab.java
import java.util.concurrent.StructuredTaskScope;
import java.time.Duration;

public class StructuredLab {
    public static void main(String[] args) throws Exception {
        try (var scope = StructuredTaskScope.open()) {
            var userTask = scope.fork(() -> fetchUser());
            var orderTask = scope.fork(() -> fetchOrder());

            scope.join();

            System.out.println("Combined: " + userTask.get() + " | " + orderTask.get());
        }
    }

    static String fetchUser() throws InterruptedException {
        System.out.println("Fetching user on " + Thread.currentThread());
        Thread.sleep(Duration.ofSeconds(1));
        return "User(name=Alex)";
    }

    static String fetchOrder() throws InterruptedException {
        System.out.println("Fetching order on " + Thread.currentThread());
        Thread.sleep(Duration.ofSeconds(2));
        return "Order(id=123)";
    }
}

In [ ]:
!javac --enable-preview --source 25 StructuredLab.java && java --enable-preview StructuredLab

---
# Module 5: Unnamed Variables & Markdown JavaDoc (Java 22-23)

**Goal**: Write cleaner code by discarding unused variables, and adopt modern documentation syntax.

## 5.1 Unnamed Variables and Patterns (Java 22, JEP 456)

Use `_` wherever a variable is required by syntax but never read.

In [ ]:
%%writefile UnnamedVarLab.java
import java.util.List;
import java.util.Map;
import java.util.stream.Collectors;

public class UnnamedVarLab {

    sealed interface Shape permits Circle, Rectangle, Triangle {}
    record Circle(double radius) implements Shape {}
    record Rectangle(double w, double h) implements Shape {}
    record Triangle(double base, double height) implements Shape {}

    public static void main(String[] args) {
        // In enhanced for-loops: counting without using the element
        var items = List.of("a", "b", "c");
        int count = 0;
        for (var _ : items) { count++; }
        System.out.println("Count: " + count);

        // In catch blocks
        try {
            Integer.parseInt("not-a-number");
        } catch (NumberFormatException _) {
            System.out.println("Invalid number format");
        }

        // In lambdas: unused parameter
        Map<String, String> lookup = items.stream()
            .collect(Collectors.toMap(String::toUpperCase, _ -> "UNKNOWN"));
        System.out.println("Lookup: " + lookup);

        // In pattern matching with switch
        Shape shape = new Circle(5.0);
        String description = switch (shape) {
            case Circle _    -> "It's a circle";
            case Rectangle _ -> "It's a rectangle";
            case Triangle _  -> "It's a triangle";
        };
        System.out.println(description);
    }
}

In [ ]:
!javac UnnamedVarLab.java && java UnnamedVarLab

## 5.2 Markdown Documentation Comments (Java 23, JEP 467)

JavaDoc can now be written in **Markdown** using `///` line comments:

```java
/// Returns the greater of two `int` values.
///
/// ## Example
/// ```
/// int max = MathUtils.max(3, 7); // returns 7
/// ```
///
/// @param a the first operand
/// @param b the second operand
/// @return the greater of `a` and `b`
public static int max(int a, int b) {
    return (a >= b) ? a : b;
}
```

The `@param`, `@return`, `@throws` tags still work inside `///` blocks. You can mix Markdown with traditional tags.

---
# Module 6: Stream Gatherers & Scoped Values (Java 24-25)

**Goal**: Extend streams with custom intermediate operations and replace `ThreadLocal` safely.

## 6.1 Stream Gatherers (Java 24, JEP 485)

`gather()` is a new intermediate stream operation for custom processing. Built-in gatherers ship in `java.util.stream.Gatherers`.

In [ ]:
%%writefile GathererLab.java
import java.util.List;
import java.util.stream.Gatherers;

public class GathererLab {
    public static void main(String[] args) {
        List<String> items = List.of("A", "B", "C", "D", "E");

        // Fixed-size windows: group elements into chunks
        var windows = items.stream()
            .gather(Gatherers.windowFixed(3))
            .toList();
        System.out.println("Fixed windows: " + windows);
        // [[A, B, C], [D, E]]

        // Sliding windows: overlapping groups
        var sliding = items.stream()
            .gather(Gatherers.windowSliding(3))
            .toList();
        System.out.println("Sliding windows: " + sliding);
        // [[A, B, C], [B, C, D], [C, D, E]]

        // Scan: running accumulation (emits each intermediate result)
        List<Integer> transactions = List.of(1000, -200, -500, 200, -300);
        var balanceHistory = transactions.stream()
            .gather(Gatherers.scan(() -> 0, Integer::sum))
            .toList();
        System.out.println("Balance history: " + balanceHistory);
        // [1000, 800, 300, 500, 200]

        // Fold: reduce-like but as a gatherer
        String concatenated = items.stream()
            .gather(Gatherers.fold(() -> "", (acc, el) -> acc + el))
            .findFirst().orElse("");
        System.out.println("Folded: " + concatenated);
        // ABCDE
    }
}

In [ ]:
!javac GathererLab.java && java GathererLab

## 6.2 Scoped Values (Java 25, JEP 506)

`ScopedValue` replaces `ThreadLocal` with an immutable, scope-bounded, auto-cleaned alternative.

| | `ThreadLocal` | `ScopedValue` |
|---|---|---|
| Mutability | Mutable (`set()` anytime) | Immutable per scope |
| Cleanup | Manual (`remove()` or leak) | Automatic when scope ends |
| Virtual threads | `InheritableThreadLocal` (copies) | Native support |
| Performance | HashMap per thread | Optimized for short-lived scopes |

In [ ]:
%%writefile ScopedValueLab.java
public class ScopedValueLab {

    static final ScopedValue<String> CURRENT_USER = ScopedValue.newInstance();

    public static void main(String[] args) {
        ScopedValue.where(CURRENT_USER, "alice").run(() -> {
            processOrder();
            sendConfirmation();
        });
        // CURRENT_USER is no longer bound here

        ScopedValue.where(CURRENT_USER, "bob").run(() -> {
            processOrder();
        });
    }

    static void processOrder() {
        System.out.println("Processing order for: " + CURRENT_USER.get());
    }

    static void sendConfirmation() {
        System.out.println("Sending confirmation to: " + CURRENT_USER.get());
    }
}

In [ ]:
!javac ScopedValueLab.java && java ScopedValueLab

---
# Module 7: Java 25 Language Features (Java 25 LTS)

**Goal**: Learn the three new language features that simplify everyday Java code.

## 7.1 Module Import Declarations (JEP 511)

Import all packages exported by a module with a single `import module` statement.

In [ ]:
%%writefile ModuleImportLab.java
import module java.base;

// No need for individual imports of List, Map, Collectors, etc.
public class ModuleImportLab {
    public static void main(String[] args) {
        List<String> names = List.of("Alice", "Bob", "Charlie");
        Map<String, Integer> nameLengths = names.stream()
            .collect(Collectors.toMap(n -> n, String::length));
        System.out.println("Name lengths: " + nameLengths);

        // java.time is also part of java.base
        System.out.println("Today: " + LocalDate.now());
    }
}

In [ ]:
!javac ModuleImportLab.java && java ModuleImportLab

## 7.2 Compact Source Files & Instance Main Methods (JEP 512)

Write simple programs without `public class`, `static`, or `String[] args`.

The launch protocol searches for `main()` in this order:
1. `static void main(String[])` — traditional
2. `static void main()` — no-args static
3. `void main(String[])` — instance with args
4. `void main()` — instance no-args

In [ ]:
%%writefile CompactHello.java
// A complete Java 25 program. No class declaration, no static, no String[] args.
void main() {
    var names = java.util.List.of("Alice", "Bob", "Charlie");
    names.stream()
        .filter(n -> n.length() > 3)
        .forEach(n -> println("Hello, " + n + "!"));
}

In [ ]:
!javac CompactHello.java && java CompactHello

## 7.3 Flexible Constructor Bodies (JEP 513)

Statements can now appear **before** `super()` or `this()`. No more static helper methods for argument validation.

In [ ]:
%%writefile FlexConstructorLab.java
public class FlexConstructorLab {

    static class Person {
        private final String name;
        Person(String name) { this.name = name; }
        String name() { return name; }
    }

    static class Employee extends Person {
        private final String department;

        Employee(String name, String department) {
            // Prologue: validate and transform BEFORE calling super()
            if (name == null || name.isBlank()) {
                throw new IllegalArgumentException("Name is required");
            }
            var normalizedName = name.strip().toUpperCase();

            super(normalizedName);  // pass computed value to super
            this.department = department;
        }
    }

    public static void main(String[] args) {
        var emp = new Employee("  alice  ", "Engineering");
        System.out.println("Name: " + emp.name());
        System.out.println("Dept: " + emp.department);

        try { new Employee("  ", "HR"); }
        catch (IllegalArgumentException e) { System.out.println("Caught: " + e.getMessage()); }
    }
}

In [ ]:
!javac FlexConstructorLab.java && java FlexConstructorLab

---
# Final Master Challenge

**Scenario**: Build a notification system.

1. Create a `Notification` record with `user`, `message`, `isUrgent`
2. Use a Stream to **partition** into urgent/non-urgent
3. **Group** urgent notifications by user
4. Use a **Virtual Thread Executor** to process each user's urgent messages concurrently

In [ ]:
%%writefile MasterChallenge.java
import java.util.List;
import java.util.Map;
import java.util.concurrent.Executors;
import java.util.stream.Collectors;
import java.time.Duration;

public class MasterChallenge {

    record Notification(String user, String message, boolean isUrgent) {}

    public static void main(String[] args) {
        List<Notification> notifications = List.of(
            new Notification("user1", "Package shipped.", false),
            new Notification("user2", "Security alert: new login.", true),
            new Notification("user1", "Your subscription is ending.", true),
            new Notification("user3", "Weekly newsletter.", false),
            new Notification("user2", "Your invoice is ready.", true)
        );

        Map<Boolean, List<Notification>> partitioned = notifications.stream()
            .collect(Collectors.partitioningBy(Notification::isUrgent));

        Map<String, List<Notification>> urgentByUser = partitioned.get(true).stream()
            .collect(Collectors.groupingBy(Notification::user));

        System.out.println("Processing urgent notifications...");
        try (var executor = Executors.newVirtualThreadPerTaskExecutor()) {
            urgentByUser.forEach((user, userNotifs) -> {
                executor.submit(() -> {
                    System.out.printf("--- Sending %d urgent to %s on %s ---\n",
                        userNotifs.size(), user, Thread.currentThread());
                    userNotifs.forEach(n -> {
                        try {
                            Thread.sleep(Duration.ofMillis(500));
                            System.out.printf("  -> SENT to %s: %s\n", user, n.message().toUpperCase());
                        } catch (InterruptedException e) {}
                    });
                });
            });
        }
        System.out.println("All urgent notifications sent!");
    }
}

In [ ]:
!javac MasterChallenge.java && java MasterChallenge

---
# 30-Minute Advanced Stream & Lambda Challenge

**Goal**: Complex e-commerce queries. Try solving each task BEFORE running the solution!

| # | Goal | Key operations |
|---|---|---|
| 1 | Count "Books" products | `filter`, `count` |
| 2 | Find most recent order | `max`, `Comparator.comparing` |
| 3 | Get order 1002 status or "not found" | `filter`, `findFirst`, `map`, `orElse` |
| 4 | Unique products from Tier 1 customers | `filter`, `flatMap`, `distinct` |
| 5 | Map: orderId -> Set of products | `Collectors.toMap` |
| 6 | Sort: category ASC, price DESC | `sorted`, `Comparator.thenComparing` |
| 7 | Total value across all orders | `flatMap`, `mapToDouble`, `reduce`/`sum` |

In [ ]:
%%writefile EcommerceChallenge.java
import java.time.LocalDate;
import java.util.*;
import java.util.stream.Collectors;

public class EcommerceChallenge {

    record Customer(long id, String name, int tier) {}
    record Product(long id, String name, String category, double price) {}
    record Order(long id, LocalDate orderDate, LocalDate deliveryDate, String status, long customerId, Set<Product> products) {}

    public static void main(String[] args) {
        Customer cust1 = new Customer(1L, "Alice", 1);
        Customer cust2 = new Customer(2L, "Bob", 2);
        Customer cust3 = new Customer(3L, "Charlie", 1);

        Product prod1 = new Product(101L, "Laptop", "Electronics", 1200.00);
        Product prod2 = new Product(102L, "Desk Chair", "Furniture", 350.50);
        Product prod3 = new Product(103L, "Java 8 in Action", "Books", 45.99);
        Product prod4 = new Product(104L, "Monitor", "Electronics", 499.99);
        Product prod5 = new Product(105L, "Clean Code", "Books", 38.99);

        Order order1 = new Order(1001L, LocalDate.of(2023, 10, 15), LocalDate.of(2023, 10, 20), "DELIVERED", 1L, Set.of(prod1, prod3));
        Order order2 = new Order(1002L, LocalDate.of(2023, 11, 1), LocalDate.of(2023, 11, 5), "PROCESSING", 2L, Set.of(prod2, prod4));
        Order order3 = new Order(1003L, LocalDate.of(2023, 10, 28), LocalDate.of(2023, 11, 2), "DELIVERED", 1L, Set.of(prod3, prod5));
        Order order4 = new Order(1004L, LocalDate.of(2023, 11, 3), LocalDate.of(2023, 11, 6), "SHIPPED", 3L, Set.of(prod5));

        List<Customer> customers = List.of(cust1, cust2, cust3);
        List<Product> products = List.of(prod1, prod2, prod3, prod4, prod5);
        List<Order> orders = List.of(order1, order2, order3, order4);

        System.out.println("--- Task 1: Count Books ---");
        long bookCount = products.stream()
            .filter(p -> "Books".equals(p.category())).count();
        System.out.println("Number of books: " + bookCount);

        System.out.println("\n--- Task 2: Most Recent Order ---");
        var mostRecent = orders.stream().max(Comparator.comparing(Order::orderDate));
        System.out.println("Most recent: " + mostRecent);

        System.out.println("\n--- Task 3: Find Order 1002 ---");
        String status = orders.stream().filter(o -> o.id() == 1002L)
            .findFirst().map(Order::status).orElse("Order not found");
        System.out.println("Status: " + status);

        System.out.println("\n--- Task 4: Tier 1 Products ---");
        Set<Long> tier1Ids = customers.stream().filter(c -> c.tier() == 1)
            .map(Customer::id).collect(Collectors.toSet());
        var tier1Products = orders.stream()
            .filter(o -> tier1Ids.contains(o.customerId()))
            .flatMap(o -> o.products().stream()).distinct().toList();
        System.out.println("Products: " + tier1Products);

        System.out.println("\n--- Task 5: Order to Products Map ---");
        Map<Long, Set<Product>> orderMap = orders.stream()
            .collect(Collectors.toMap(Order::id, Order::products));
        System.out.println("Map: " + orderMap);

        System.out.println("\n--- Task 6: Sorted Products ---");
        products.stream()
            .sorted(Comparator.comparing(Product::category)
                .thenComparing(Product::price, Comparator.reverseOrder()))
            .forEach(p -> System.out.println("  " + p));

        System.out.println("\n--- Task 7: Total Order Value (reduce) ---");
        double total = orders.stream().flatMap(o -> o.products().stream())
            .mapToDouble(Product::price).sum();
        System.out.println("Total value: " + total);
    }
}

In [ ]:
!javac EcommerceChallenge.java && java EcommerceChallenge

---
# 30-Minute Modern Java Challenge (Java 22-25)

**Goal**: Practice Java 22-25 features in a realistic IoT sensor monitoring scenario.

| # | Goal | Key features |
|---|---|---|
| 1 | Classify sensor readings | Unnamed patterns `_`, switch |
| 2 | Sliding window moving average | `Gatherers.windowSliding` |
| 3 | Batch readings into fixed windows | `Gatherers.windowFixed` |
| 4 | Running temperature delta | `Gatherers.scan` |
| 5 | Constructor validation before `super()` | Flexible constructor bodies |
| 6 | Scoped analysis session | `ScopedValue` |

## Setup: Data Models and Test Data

Try solving each task in the `main` method BEFORE looking at the solution below!

In [ ]:
%%writefile SensorSetup.java
import module java.base;

public class SensorSetup {

    // --- Sealed hierarchy for sensor readings ---
    sealed interface SensorReading permits Temperature, Humidity, Pressure {}
    record Temperature(String sensorId, double celsius, long timestamp) implements SensorReading {}
    record Humidity(String sensorId, double percent, long timestamp) implements SensorReading {}
    record Pressure(String sensorId, double hPa, long timestamp) implements SensorReading {}

    // --- Sensor with constructor to modify in Task 5 ---
    static class Sensor {
        private final String id;
        private final String location;
        Sensor(String id, String location) {
            // Task 5: Add validation BEFORE field assignments
            this.id = id;
            this.location = location;
        }
        String id() { return id; }
        String location() { return location; }
    }

    public static void main(String[] args) {
        List<SensorReading> readings = List.of(
            new Temperature("T1", 22.5, 1000), new Humidity("H1", 45.0, 1000), new Pressure("P1", 1013.25, 1000),
            new Temperature("T1", 23.1, 2000), new Humidity("H1", 47.0, 2000), new Pressure("P1", 1012.80, 2000),
            new Temperature("T1", 24.8, 3000), new Humidity("H1", 50.0, 3000), new Pressure("P1", 1011.50, 3000),
            new Temperature("T1", 26.2, 4000), new Humidity("H1", 55.0, 4000), new Pressure("P1", 1010.00, 4000)
        );

        System.out.println("Setup OK. " + readings.size() + " readings loaded.");
        System.out.println("Now solve Tasks 1-6 in the cells below!");
    }
}

In [ ]:
!javac SensorSetup.java && java SensorSetup

## Solution: All 6 Tasks

Run the cell below to see the full solution. **Try solving it yourself first!**

In [ ]:
%%writefile SensorChallenge.java
import module java.base;

public class SensorChallenge {

    sealed interface SensorReading permits Temperature, Humidity, Pressure {}
    record Temperature(String sensorId, double celsius, long timestamp) implements SensorReading {}
    record Humidity(String sensorId, double percent, long timestamp) implements SensorReading {}
    record Pressure(String sensorId, double hPa, long timestamp) implements SensorReading {}

    // Task 5: Flexible constructor with validation BEFORE field assignment
    static class Sensor {
        private final String id;
        private final String location;
        Sensor(String id, String location) {
            if (id == null || id.isBlank())
                throw new IllegalArgumentException("Sensor id is required");
            if (location == null || location.isBlank())
                throw new IllegalArgumentException("Sensor location is required");
            this.id = id.strip();
            this.location = location.strip();
        }
        String id() { return id; }
        String location() { return location; }
    }

    // Task 6: Scoped value
    static final ScopedValue<String> ANALYSIS_SESSION = ScopedValue.newInstance();

    public static void main(String[] args) {
        List<SensorReading> readings = List.of(
            new Temperature("T1", 22.5, 1000), new Humidity("H1", 45.0, 1000), new Pressure("P1", 1013.25, 1000),
            new Temperature("T1", 23.1, 2000), new Humidity("H1", 47.0, 2000), new Pressure("P1", 1012.80, 2000),
            new Temperature("T1", 24.8, 3000), new Humidity("H1", 50.0, 3000), new Pressure("P1", 1011.50, 3000),
            new Temperature("T1", 26.2, 4000), new Humidity("H1", 55.0, 4000), new Pressure("P1", 1010.00, 4000)
        );

        // --- Task 1: Classify with pattern matching ---
        System.out.println("--- Task 1: Classify Readings ---");
        readings.forEach(r -> {
            String summary = switch (r) {
                case Temperature t -> "Temperature: " + t.celsius() + " C";
                case Humidity h    -> "Humidity: " + h.percent() + " %";
                case Pressure p    -> "Pressure: " + p.hPa() + " hPa";
            };
            System.out.println(summary);
        });

        // --- Task 2: Sliding window moving average ---
        System.out.println("\n--- Task 2: Sliding Window Temperature Average ---");
        var movingAvg = readings.stream()
            .filter(r -> r instanceof Temperature)
            .map(r -> ((Temperature) r).celsius())
            .gather(Gatherers.windowSliding(3))
            .map(w -> w.stream().mapToDouble(Double::doubleValue).average().orElse(0.0))
            .toList();
        System.out.println("Moving averages (window=3): " + movingAvg);

        // --- Task 3: Fixed window batching ---
        System.out.println("\n--- Task 3: Fixed Window Batches ---");
        var batches = readings.stream()
            .gather(Gatherers.windowFixed(3))
            .toList();
        for (int i = 0; i < batches.size(); i++) {
            System.out.println("Batch " + (i + 1) + ": " + batches.get(i));
        }

        // --- Task 4: Running delta with scan ---
        System.out.println("\n--- Task 4: Running Temperature Delta ---");
        var temps = readings.stream()
            .filter(r -> r instanceof Temperature)
            .map(r -> ((Temperature) r).celsius())
            .toList();

        var deltas = temps.stream()
            .gather(Gatherers.scan(
                () -> new double[]{0.0, Double.NaN},
                (state, temp) -> {
                    double delta = Double.isNaN(state[1]) ? 0.0 : temp - state[1];
                    return new double[]{delta, temp};
                }
            ))
            .map(s -> Math.round(s[0] * 10.0) / 10.0)
            .toList();
        System.out.println("Deltas: " + deltas);

        // --- Task 5: Flexible constructor ---
        System.out.println("\n--- Task 5: Flexible Constructor Validation ---");
        var sensor = new Sensor("T1", "Server Room A");
        System.out.println("Created: " + sensor.id() + " @ " + sensor.location());
        try { new Sensor("", "Rooftop"); }
        catch (IllegalArgumentException e) { System.out.println("Caught: " + e.getMessage()); }

        // --- Task 6: Scoped Values ---
        System.out.println("\n--- Task 6: Scoped Values ---");
        ScopedValue.where(ANALYSIS_SESSION, "Session-A").run(() -> analyzeReadings(readings));
        ScopedValue.where(ANALYSIS_SESSION, "Session-B").run(() -> analyzeReadings(readings));
    }

    static void analyzeReadings(List<SensorReading> readings) {
        System.out.println("[" + ANALYSIS_SESSION.get() + "] Analyzing " + readings.size() + " readings");
    }
}

In [ ]:
!javac SensorChallenge.java && java SensorChallenge